In [3]:
import pandas as pd, textstat, json
from readability import Readability

#### Readability

In [ ]:
with open("xaiqb_human.json") as f:
    xaiqb_human = json.load(f)

with open("xaiqb_chatgpt.json") as f:
    xaiqb_chatgpt = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'xaiqb_chatgpt.json'

In [4]:
def readabililty_scores(text, df = True):
    # https://readabilityformulas.com/
    # https://pypi.org/project/py-readability-metrics/
    
    if textstat.textstat.lexicon_count(text) >= 100:
        r = Readability(text)

        scores = {
            "text": text,
            # "stats": r.statistics(),
            "flesch_kincaid": {
                "score": r.flesch_kincaid().score, 
                "grade_level": r.flesch_kincaid().grade_level,
            },
            "flesch_reading": {
                "score": r.flesch().score,
                "ease": r.flesch().ease, 
                "grade_level": r.flesch().grade_levels,
            },
            "dale_chall": {
                "score": r.dale_chall().score,
                "grade_level": r.dale_chall().grade_levels,
            },
            "ari": {
                "score": r.ari().score,
                "grade_level": r.ari().grade_levels,
                "ages": r.ari().ages
            },
            "coleman_liau": {
                "score": r.coleman_liau().score, 
                "grade_level": r.coleman_liau().grade_level
            },
            "gunning_fog": {
                "score": r.gunning_fog().score, 
                "grade_level": r.gunning_fog().grade_level
            },
            "smog": {
                "score": r.smog().score, 
                "grade_level": r.smog().grade_level
            },
            "spache": {
                "score": r.spache().score, 
                "grade_level": r.spache().grade_level
            },
            "linsear_write": {
                "score": r.linsear_write().score, 
                "grade_level": r.linsear_write().grade_level
            }
        }
    else:
        scores = {
            "flesch_reading_ease": textstat.flesch_reading_ease(text),
            "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
            "smog_index": textstat.smog_index(text),
            "coleman_liau_index": textstat.coleman_liau_index(text),
            "automated_readability_index": textstat.automated_readability_index(text),
            "dale_chall_readability_score": textstat.dale_chall_readability_score(text),
            # "difficult_words":  textstat.difficult_words(text),
            "linsear_write_formula": textstat.linsear_write_formula(text),
            "gunning_fog": textstat.gunning_fog(text),
            "text_standard": textstat.text_standard(text),
            "spache_readability": textstat.spache_readability(text)
            }

    return pd.DataFrame(scores).drop(["text"], axis = 1).drop(["ease", "ages"], axis = 0) if df else scores

In [ ]:
readabililty_scores(
    
)

,flesch_kincaid,flesch_reading,dale_chall,ari,coleman_liau,gunning_fog,smog,spache,linsear_write
score,5.459759,71.995496,8.66009,5.045697,8.234633,4.977095,10.355216,5.267603,5.166667
grade_level,5,[7],"[11, 12]",[6],8,na,10,5,5


In [25]:
readabililty_scores(
    "This is important for me to understand how this works",
    df = False
)

{'flesch_reading_ease': 78.24500000000002,
 'flesch_kincaid_grade': 4.830000000000002,
 'smog_index': 11.20814326018867,
 'coleman_liau_index': 6.760000000000002,
 'automated_readability_index': 4.2940000000000005,
 'dale_chall_readability_score': 5.7115,
 'linsear_write_formula': 6.0,
 'gunning_fog': 4.0,
 'text_standard': '3rd and 4th grade',
 'spache_readability': 2.2489999999999997}

##### Similarity

https://spotintelligence.com/2022/12/19/text-similarity-python/#1_Text_similarity_with_NLTK

In [10]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
t1 = "Hello, my name is Henry and I am an MSc. Data Science Student at FU Berlin. I am currently writing my thesis on Large Language Models"
t2 = "Hello, my name is Alison and I work in a neuropsychology clinic just outside of Berlin."

In [45]:
# Convert the texts into TF-IDF vectors
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform([t1, t2])

# Calculate the cosine similarity between the vectors
similarity = cosine_similarity(vectors)
print(similarity)

[[1.         0.37997836]
 [0.37997836 1.        ]]


In [46]:
t1 = "The data came from the H1N1 survey conducted in 2009 by the US National Center for Health Statistics. This took place over the phone."
t2 = "US National Center for Health Statistics"

In [47]:
import transformers
# Load the BERT model
# model = transformers.BertModel.from_pretrained('bert-base-uncased')
# Load the RoBERTa model
# model = transformers.RobertaModel.from_pretrained('roberta-base')
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# Tokenize and encode the texts
encoding1 = model.encode(t1, normalize_embeddings = True)
encoding2 = model.encode(t2, normalize_embeddings = True)

# Calculate the cosine similarity between the embeddings
similarity = np.dot(encoding1, encoding2) / (np.linalg.norm(encoding1) * np.linalg.norm(encoding2))
print(similarity)

0.58778286


In [ ]:
# https://www.newscatcherapi.com/blog/ultimate-guide-to-text-similarity-with-python

In [50]:
import torch

def cosine_similarity(text1, text2):
  # Convert the texts to tensors
  text1 = torch.tensor([text1])
  text2 = torch.tensor([text2])
  
  # Calculate the dot product of the texts
  dot_product = torch.matmul(text1, text2.transpose(1, 0))

  # Calculate the norms of the texts
  norm1 = torch.norm(text1, dim=1)
  norm2 = torch.norm(text2, dim=1)

  # Calculate the cosine similarity
  cosine_similarity = dot_product / (norm1 * norm2)

  return cosine_similarity

similarity = cosine_similarity(t1, t2)

print(f"Similarity between t1 and t2: {similarity:.2f}")

ValueError: too many dimensions 'str'

#### Readability Metric Descriptions

##### Flesch-Kincaid Grade Level
The U.S. Army uses Flesch-Kincaid Grade Level for assessing the difficulty of technical manuals. The commonwealth of Pennsylvania uses Flesch-Kincaid Grade Level for scoring automobile insurance policies to ensure their texts are no higher than a ninth grade level of reading difficulty. Many other U.S. states also use Flesch-Kincaid Grade Level to score other legal documents such as business policies and financial forms.

##### Flesch Reading Ease
The U.S. Department of Defense uses the Reading Ease test as the standard test of readability for its documents and forms. Florida requires that life insurance policies have a Flesch Reading Ease score of 45 or greater.

##### Dale Chall Readability
The Dale-Chall Formula is an accurate readability formula for the simple reason that it is based on the use of familiar words, rather than syllable or letter counts. Reading tests show that readers usually find it easier to read, process and recall a passage if they find the words familiar.

##### Automated Readability Index (ARI)
Unlike the other indices, the ARI, along with the Coleman-Liau, relies on a factor of characters per word, instead of the usual syllables per word. ARI is widely used on all types of texts.

##### Coleman Liau Index
The Coleman-Liau Formula usually gives a lower grade value than any of the Kincaid, ARI and Flesch values when applied to technical documents.

##### Gunning Fog
The Gunning fog index measures the readability of English writing. The index estimates the years of formal education needed to understand the text on a first reading. A fog index of 12 requires the reading level of a U.S. high school senior (around 18 years old).

##### SMOG
The SMOG Readability Formula (Simple Measure of Gobbledygook) is a popular method to use on health literacy materials.

##### SPACHE
The Spache Readability Formula is used for Primary-Grade Reading Materials, published in 1953 in The Elementary School Journal. The Spache Formula is best used to calculate the difficulty of text that falls at the 3rd grade level or below.

##### Linsear Write
Linsear Write is a readability metric for English text, purportedly developed for the United States Air Force to help them calculate the readability of their technical manuals.